# 03 - Investigação de perguntas potencialmente condicionadas

## Objetivo e âmbito

Investigar `IN22`, `PC1B`, `HA1A` e `IN15` por `AGE_RANGE` e `SEX`, com atenção às respostas explícitas `Não aplicável` e `Sem resposta`. A exploração usa **apenas a parcela de treino** da divisão estratificada fixa de 80/20 (`random_state=42`); não lê nem consulta o CSV de teste Kaggle. Não são ajustados modelos neste notebook.

Segundo o enunciado da competição, `AGE_RANGE=8` corresponde a 50–54 anos e `AGE_RANGE=9` a 55–59 anos. Esta correspondência é tratada como informação do enunciado, não inferida das respostas.

As percentagens descrevem associações observadas. Uma concentração de `Não aplicável` pode ser compatível com uma regra de elegibilidade/skip pattern, mas não é, por si só, prova de leakage nem de uma regra documentada. A única experiência com modelos neste notebook é a ablação por validação cruzada descrita no final; não é consultada a validação reservada nem o teste.


## Fontes do questionário e limites da documentação

Não há questionário, codebook ou dicionário de variáveis nos ficheiros locais do projeto. Foi consultada a descrição oficial do [Eurostat - European Health Interview Survey (EHIS)](https://ec.europa.eu/eurostat/web/microdata/collections-research/european-health-interview-survey), que descreve o âmbito geral da pesquisa e as variáveis de contexto, e o [Regulamento (UE) 2018/255](https://eur-lex.europa.eu/eli/reg/2018/255/oj/eng), que estabelece a recolha EHIS. Estas fontes confirmam o contexto geral do inquérito, mas não fornecem, no material consultado, o mapeamento dos códigos locais `IN22`, `PC1B`, `HA1A`, `IN15` para texto integral e condições de aplicação.

Assim, as observações abaixo **não são classificadas como regras documentadas**. Para confirmar texto, universos elegíveis e filtros, é necessário obter o questionário/código de variáveis correspondente à versão nacional e à vaga destes dados. A indicação de que a fronteira de interesse é 50–54/55–59 vem do enunciado da competição.


## Carregar apenas o treino e fixar a parcela de exploração

A divisão é feita no treino original antes da preparação fixa. O relatório estatístico usa apenas as linhas da parcela de treino; a parcela de validação não é inspecionada nesta investigação. A preparação fixa não aprende imputações, categorias ou outros parâmetros.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.preparar_dados import make_stratified_split, preparar_dados_fixos

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
if not train_path.is_file():
    raise FileNotFoundError(f'Ficheiro de treino não encontrado: {train_path}')

source_train = pd.read_csv(train_path)
if 'AGE_RANGE' not in source_train or 'SEX' not in source_train:
    raise ValueError('O treino tem de incluir AGE_RANGE e SEX para esta análise.')

X_raw = source_train.drop(columns=['AGE_RANGE'])
y_raw = source_train['AGE_RANGE']
X_train_raw, X_val_raw, y_train, y_val = make_stratified_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

training_partition = source_train.loc[X_train_raw.index].copy()
fixed = preparar_dados_fixos(training_partition)
analysis = fixed['X'].copy()
analysis['AGE_RANGE'] = fixed['y']
analysis['SEX'] = training_partition['SEX']

variables = ['IN22', 'PC1B', 'HA1A', 'IN15']
missing_variables = [column for column in variables if column not in analysis.columns]
if missing_variables:
    raise ValueError(f'Variáveis em falta no treino: {missing_variables}')

assert len(analysis) == len(y_train) == 8769
assert analysis.index.equals(y_train.index)
print(f'Parcela de treino usada: {len(analysis)} linhas; validação reservada: {len(X_val_raw)} linhas.')
print(f'Classes observadas: {sorted(analysis["AGE_RANGE"].unique().tolist())}')
print(f'Categorias de SEX: {analysis["SEX"].value_counts(dropna=False).to_dict()}')
print('AGE_RANGE 8=50–54 e 9=55–59 conforme o enunciado.')


## Distribuições por `AGE_RANGE` e por `SEX`

Para cada variável, mostramos contagens e percentagens dentro de cada classe, primeiro no total e depois separadas por sexo. As colunas incluem todas as respostas observadas e mantêm `Não aplicável`, `Sem resposta` e ausências de origem como categorias distintas. A percentagem é calculada dentro de cada linha (classe, ou classe e sexo), pelo que as linhas somam 100% salvo arredondamento.


In [ ]:
SPECIAL_RESPONSES = ['Não aplicável', 'Sem resposta', 'Ausente de origem (NaN)']

def response_tables(frame, variable):
    response = frame[variable].astype('string').fillna('Ausente de origem (NaN)')
    observed = list(pd.unique(response))
    columns = list(dict.fromkeys(SPECIAL_RESPONSES + observed))
    age = frame['AGE_RANGE'].astype(int)
    sex = frame['SEX'].astype('string').fillna('Ausente de origem (NaN)')

    counts_by_age = pd.crosstab(age, response).reindex(columns=columns, fill_value=0)
    pct_by_age = counts_by_age.div(counts_by_age.sum(axis=1), axis=0).mul(100)

    counts_by_age_sex = pd.crosstab([age, sex], response).reindex(columns=columns, fill_value=0)
    pct_by_age_sex = counts_by_age_sex.div(counts_by_age_sex.sum(axis=1), axis=0).mul(100)
    return counts_by_age, pct_by_age, counts_by_age_sex, pct_by_age_sex

for variable in variables:
    print(f'\n### {variable}')
    print('Valores ausentes reais na parcela de treino:', int(analysis[variable].isna().sum()))
    counts_age, pct_age, counts_age_sex, pct_age_sex = response_tables(analysis, variable)
    print('Contagens por AGE_RANGE')
    display(counts_age)
    print('Percentagens por AGE_RANGE (%)')
    display(pct_age.round(1))
    print('Contagens por AGE_RANGE e SEX')
    display(counts_age_sex)
    print('Percentagens por AGE_RANGE e SEX (%)')
    display(pct_age_sex.round(1))


## Zoom na fronteira 50–54 / 55–59

A tabela seguinte permite comparar diretamente as classes 8 e 9 por sexo e resposta, sem agregar `Não aplicável` com `Sem resposta`. É importante considerar os tamanhos de cada subgrupo ao interpretar percentagens.


In [ ]:
boundary = analysis.loc[analysis['AGE_RANGE'].isin([8, 9])].copy()
for variable in variables:
    response = boundary[variable].astype('string').fillna('Ausente de origem (NaN)')
    counts = pd.crosstab(
        [boundary['AGE_RANGE'], boundary['SEX']], response
    )
    percentages = counts.div(counts.sum(axis=1), axis=0).mul(100)
    print(f'\n{variable} — contagens (classes 8=50–54 e 9=55–59)')
    display(counts)
    print(f'{variable} — percentagens dentro de AGE_RANGE e SEX (%)')
    display(percentages.round(1))


## Leitura dos padrões observados

Na parcela de treino usada (8.769 pessoas), `PC1B`, `HA1A` e `IN15` têm `Não aplicável` em **100% das observações das classes 0–8**, tanto em homens como em mulheres. As respostas substantivas começam na classe 9 (55–59), para ambos os sexos. Isto é um indício muito forte de uma condição de aplicação associada ao escalão etário, e não apenas a uma diferença na composição por sexo. Não demonstra, contudo, a regra oficial nem esclarece se a condição é idade exata, universo do módulo ou outra elegibilidade perfeitamente correlacionada com a idade.

Para `IN22`, o padrão difere: os homens têm `Não aplicável` nesta variável em todos os escalões observados; entre mulheres, a distribuição varia com o escalão. Na classe 8 (50–54), 35,5% respondem `Está na menopausa` e 27,3% `Não aplicável`; na classe 9 (55–59), 84,3% respondem `Não aplicável` e 7,9% `Está na menopausa`. A partir da classe 10 (60–64, de acordo com a sequência de escalões do enunciado, se confirmada), a análise total mostra apenas `Não aplicável`. Este padrão sugere fortemente uma dimensão de sexo e idade/eligibilidade, mas a pergunta e a regra não podem ser determinadas só pelas categorias.

Em `HA1A`, há também diferenças de sexo dentro da classe 9: `Não aplicável (nunca tentou ou não precisa de o fazer)` é mais frequente entre homens do que mulheres. Esta resposta é distinta do `Não aplicável` geral presente abaixo da classe 9 e pode refletir a própria semântica da pergunta, diferenças de experiência ou outros fatores. `PC1B` e `IN15` mostram distribuições substantivas relativamente semelhantes entre sexos na classe 9, embora os números de respostas com dificuldade sejam pequenos.

`Sem resposta` aparece com frequência baixa; as tabelas mantêm-na explícita. Não foram observados NaN reais nestas quatro variáveis nesta parcela, mas o código mantém uma categoria própria para esse caso se existir noutra versão dos dados.

**Conclusão provisória:** há padrões preditivos que podem codificar elegibilidade por idade e, no caso de `IN22`, por sexo. Não classificamos automaticamente as variáveis como leakage nem recomendamos removê-las sem consultar a documentação. Se as mesmas regras de recolha estiverem presentes no conjunto de teste, estes campos podem ser preditores legítimos; se a disponibilidade/definição diferir entre treino e teste, poderão causar fragilidade.


## Comparação emparelhada por validação cruzada

Comparamos as duas versões solicitadas usando somente as 8.769 linhas da parcela de treino. A validação reservada não é usada. Ambas recebem exatamente os mesmos índices de fold (`StratifiedKFold(n_splits=3, shuffle=True, random_state=42)`). A pipeline do 002 é mantida sem alterações; `cross_validate` clona e ajusta uma pipeline/preparação nova em cada fold. Não filtramos avisos de convergência, usamos `n_jobs=1` e calculamos a diferença emparelhada como Macro F1 (todas as variáveis) menos Macro F1 (sem as quatro).

1. **Todas as variáveis**: inclui `PC1B`, `HA1A`, `IN15` e `IN22`.
2. **Sem as quatro variáveis**: exclui apenas `PC1B`, `HA1A`, `IN15` e `IN22`; mantém `SEX` e todas as restantes entradas.

Esta comparação mede a utilidade preditiva conjunta das quatro variáveis; não substitui a confirmação da regra no questionário nem decide se as variáveis devem ser consideradas leakage.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.preparar_dados import make_model_pipeline

X_all = fixed['X'].copy()
y_cv = fixed['y'].copy()
ablation_columns = ['PC1B', 'HA1A', 'IN15', 'IN22']
missing_ablation_columns = [column for column in ablation_columns if column not in X_all.columns]
if missing_ablation_columns:
    raise ValueError(f'Colunas da ablação em falta: {missing_ablation_columns}')

X_without_questions = X_all.drop(columns=ablation_columns)
assert len(X_all) == len(X_without_questions) == len(y_cv) == 8769
assert X_all.index.equals(X_without_questions.index) and X_all.index.equals(y_cv.index)
assert 'SEX' in X_without_questions.columns

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
same_folds = list(cv.split(X_all, y_cv))
versions = {
    'Todas as variáveis': X_all,
    'Sem PC1B, HA1A, IN15 e IN22': X_without_questions,
}
fold_scores = {}

for version_name, X_version in versions.items():
    pipeline = make_model_pipeline(
        LogisticRegression(
            penalty='l2',
            C=0.1,
            class_weight=None,
            solver='lbfgs',
            max_iter=2000,
            random_state=42,
        ),
        normalize_numeric=True,
    )
    # cross_validate clones and fits the entire pipeline independently in every fold.
    # Convergence warnings are intentionally not caught or filtered.
    cv_result = cross_validate(
        pipeline,
        X_version,
        y_cv,
        cv=same_folds,
        scoring='f1_macro',
        n_jobs=1,
        error_score='raise',
    )
    fold_scores[version_name] = cv_result['test_score']

comparison = pd.DataFrame({
    'Fold': [1, 2, 3],
    'Macro F1 — todas': fold_scores['Todas as variáveis'],
    'Macro F1 — sem quatro perguntas': fold_scores['Sem PC1B, HA1A, IN15 e IN22'],
})
comparison['Diferença emparelhada (todas - sem)'] = (
    comparison['Macro F1 — todas'] - comparison['Macro F1 — sem quatro perguntas']
)
display(comparison.round(6))

summary = pd.DataFrame({
    'Versão': list(versions),
    'Média Macro F1': [fold_scores[name].mean() for name in versions],
    'Desvio padrão Macro F1': [fold_scores[name].std(ddof=0) for name in versions],
})
display(summary.round(6))
print('Diferença emparelhada média (todas - sem):', round(comparison['Diferença emparelhada (todas - sem)'].mean(), 6))
print('Desvio padrão das diferenças emparelhadas:', round(comparison['Diferença emparelhada (todas - sem)'].std(ddof=0), 6))


## Conclusão da ablação

Com a configuração do 002, a versão com todas as variáveis obteve Macro F1 por fold **0,416432**, **0,430184** e **0,427112** (média **0,424576**, desvio padrão populacional **0,005894**). Sem `PC1B`, `HA1A`, `IN15` e `IN22`, os resultados foram **0,418538**, **0,426482** e **0,425416** (média **0,423479**, desvio padrão **0,003521**). As diferenças emparelhadas (todas menos sem) foram **−0,002106**, **+0,003702** e **+0,001696**, com média **+0,001097**.

Neste benchmark, as quatro variáveis em conjunto estão associadas a um ganho médio muito pequeno de Macro F1, com resultados mistos entre folds. Isto é compatível com alguma dependência preditiva, mas não mostra que o modelo dependa fortemente delas para classificar. Os padrões de `Não aplicável` continuam a ser sinais úteis para investigar as regras de aplicação das perguntas e anomalias de perfil, especialmente a transição consistente entre classes 8 (50–54) e 9 (55–59), e a interação de `IN22` com `SEX`.

A ligeira queda ao remover as variáveis **não prova leakage**: pode refletir informação legítima, redundância com outras características ou variação amostral. Também não confirma que a presença delas seja apropriada para detetar anomalias de perfil. A avaliação usa apenas CV na parcela de treino; a validação permanece reservada para uma avaliação posterior e o teste/submissão não foram lidos nem criados.
